# Results & Error Analysis

Pulls the 256 token fine-tuned model from the Hugging Face Hub, runs it on the test
set on Colab GPU, and produces the portfolio visuals:
- verification that test Spearman matches the training run (~0.404),
- a predicted-vs-true plot,
- a 4-class confusion matrix,
- an error analysis of the model's worst predictions.

**Before you run:** set the runtime to GPU.

In [ ]:
# CONFIG
HF_MODEL_REPO = "iamahmadyasin/humor-intelligence-distilbert-256"
MAX_LENGTH    = 256
BATCH_SIZE    = 32
SAMPLE_SIZE   = None
import sys
!pip install -q transformers torch
!git clone -q https://github.com/YOUR_USERNAME/humor-intelligence.git
%cd humor-intelligence
!python src/data.py
%cd /content
sys.path.append("/content/humor-intelligence/src")

In [ ]:
# 1. Install
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2. Load the model & tokenizer from the Hub
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_REPO)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Loaded", HF_MODEL_REPO, "on", device)

In [ ]:
# 3. Load the local test split
import pandas as pd
from config import PROCESSED_DIR

test = pd.read_csv(PROCESSED_DIR / "test.csv")
if SAMPLE_SIZE:
    test = test.sample(SAMPLE_SIZE, random_state=0).reset_index(drop=True)
    print(f"Using a {SAMPLE_SIZE}-row sample")
print(f"test rows: {len(test):,}")
test.head(3)

In [ ]:
# 4. Run inference in batches.
import numpy as np
from tqdm.auto import tqdm

@torch.no_grad()
def predict(texts, batch_size=BATCH_SIZE):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        enc = tokenizer(list(batch), truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(device)
        out = model(**enc).logits.squeeze(-1)
        preds.extend(out.cpu().numpy().tolist())
    return np.array(preds)

pred = predict(test["joke"].tolist())
true = test["score"].to_numpy()
print("done! predictions:", pred.shape)

In [ ]:
# 5. Verify metrics match the training run
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

rho = spearmanr(true, pred).correlation
r   = pearsonr(true, pred)[0]
mae = mean_absolute_error(true, pred)
rmse = mean_squared_error(true, pred) ** 0.5
print(f"Test Spearman: {rho:.4f}   (training reported ~0.4059)")
print(f"Test Pearson:  {r:.4f}")
print(f"MAE:  {mae:.4f}   |   RMSE: {rmse:.4f}")

## Predicted vs. true
The red line (mean prediction per true score) should trend upward i.e. the model
tracks real humor signal. A linear regressor typically compresses toward the
mean, so expect the line to be flatter than the diagonal.

In [ ]:
import matplotlib.pyplot as plt
from config import FIG_DIR

df = pd.DataFrame({"true": true, "pred": pred})
means = df.groupby("true")["pred"].mean()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df["true"], df["pred"], alpha=0.03, s=5)
ax.plot(means.index, means.values, "o-", color="crimson",
        label="mean predicted per true score")
ax.plot([0, 11], [0, 11], "k--", alpha=0.4, label="perfect prediction")
ax.set_xlabel("true humor score")
ax.set_ylabel("predicted score")
ax.set_title(f"DistilBERT-256 predictions (Spearman {rho:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "09_pred_vs_true_256.png", dpi=150, bbox_inches="tight")
plt.show()

## Confusion matrix (4 humor classes)
Bucket both true and predicted scores into the four classes from the EDA. The
matrix is row-normalized (each row sums to 1), so cell (i, j) reads as "of jokes
truly in class i, what fraction were predicted as class j".

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

def to_bucket(s):
    s = round(float(s))
    if s <= 0: return 0
    elif s <= 2: return 1
    elif s <= 5: return 2
    else: return 3

labels = ["not funny (0)", "mild (1-2)", "funny (3-5)", "very funny (6+)"]
tb = np.array([to_bucket(x) for x in true])
pb = np.array([to_bucket(x) for x in pred])

macro_f1 = f1_score(tb, pb, average="macro")
print(f"Macro-F1 across 4 classes: {macro_f1:.3f}")

cm = confusion_matrix(tb, pb, labels=[0, 1, 2, 3])
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
ax.set_title("Confusion matrix (row-normalized)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(FIG_DIR / "09_confusion_matrix_256.png", dpi=150, bbox_inches="tight")
plt.show()

## Error analysis
The jokes the model got most wrong. Useful for the writeup: are the big misses
genuinely hard (subtle/niche humor), or are they noise in the Reddit labels
(good jokes that happened to get few upvotes, or vice versa)?

In [ ]:
df["joke"] = test["joke"].values
df["abs_err"] = (df["true"] - df["pred"]).abs()

print("Model thought these were MUCH funnier than the crowd did")
over = df.sort_values("pred", ascending=False).query("true <= 1").head(3)
for _, row in over.iterrows():
    print(f"[true {row.true:.0f}, pred {row.pred:.1f}] {row.joke[:160]}")

print("\nModel thought these were MUCH less funny than the crowd did")
under = df.sort_values("pred").query("true >= 6").head(3)
for _, row in under.iterrows():
    print(f"[true {row.true:.0f}, pred {row.pred:.1f}] {row.joke[:160]}")

## Summary

- Test Spearman matches the ~0.4059 from training, confirming the
  published model reproduces.
- The confusion matrix shows the model does well on common classes and
  struggles on the rare 'very funny' class which is the expected effect of class
  imbalance on a regression model that compresses toward the mean.
